**Load Dataset from Kaggle to Colab**

In [259]:
# %pip install Kagglehub

In [260]:
# import os
# os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_KEY

In [261]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("shubhammehta21/movie-lens-small-latest-dataset")

# print("Path to dataset files:", path)

Make a folder Dataset and copy the path

In [262]:
# import shutil
# # copy dataset from source_dir to dest_dir
# shutil.copytree(
#     "/kaggle/input/movie-lens-small-latest-dataset",
#     "/content/Dataset",
#     dirs_exist_ok=True,
# )

In [263]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


**Load dataset**

In [264]:
movies = pd.read_csv(r"D:\My Learning\ML projects\Movie-Recommendation-System\datasets\movies.csv")
ratings = pd.read_csv(r"D:\My Learning\ML projects\Movie-Recommendation-System\datasets\ratings.csv")
links = pd.read_csv(r"D:\My Learning\ML projects\Movie-Recommendation-System\datasets\links.csv")

**EDA**

In [265]:
print("Shape:", movies.shape)

Shape: (9742, 3)


In [266]:
print("Shape:", ratings.shape)

Shape: (100836, 4)


In [267]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [268]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [269]:
links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [270]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [271]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [272]:
links.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


In [273]:
movies['genres'] = movies['genres'].fillna('')

In [274]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [275]:
movies['genres'].shape

(9742,)

In [276]:
print(movies['genres'])

0       Adventure|Animation|Children|Comedy|Fantasy
1                        Adventure|Children|Fantasy
2                                    Comedy|Romance
3                              Comedy|Drama|Romance
4                                            Comedy
                           ...                     
9737                Action|Animation|Comedy|Fantasy
9738                       Animation|Comedy|Fantasy
9739                                          Drama
9740                               Action|Animation
9741                                         Comedy
Name: genres, Length: 9742, dtype: object


In [277]:
# duplicates
duplicate_mask = movies.duplicated()
num_duplicates = duplicate_mask.sum()
print("Number of duplicate rows:", num_duplicates)

# (optional) drop duplicates if present
# df = df.drop_duplicates()
# print("Shape after dropping duplicates:", df.shape)

Number of duplicate rows: 0


In [278]:
# Find duplicates 
movies[movies.duplicated(subset=['title'])]


,movieId,title,genres
5601,26958,Emma (1996),Romance
6932,64997,War of the Worlds (2005),Action|Sci-Fi
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
9135,147002,Eros (2004),Drama|Romance
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller


**Pre Processing**

In [279]:
movies = movies.drop_duplicates(subset=['title'])
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9737 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9737 non-null   int64 
 1   title    9737 non-null   object
 2   genres   9737 non-null   object
dtypes: int64(1), object(2)
memory usage: 304.3+ KB


In [280]:
# remove tmdbId from links
links = links.drop(columns=['tmdbId'])
links.head()

,movieId,imdbId
0,1,114709
1,2,113497
2,3,113228
3,4,114885
4,5,113041


In [281]:
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)
print(movies['genres'])

0       Adventure Animation Children Comedy Fantasy
1                        Adventure Children Fantasy
2                                    Comedy Romance
3                              Comedy Drama Romance
4                                            Comedy
                           ...                     
9737                Action Animation Comedy Fantasy
9738                       Animation Comedy Fantasy
9739                                          Drama
9740                               Action Animation
9741                                         Comedy
Name: genres, Length: 9737, dtype: object


In [282]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [283]:
# Compute average rating upto one decimal place and reset index
average_ratings = ratings.groupby('movieId')['rating'].mean().reset_index().round(1)

# Optional: Rename column for clarity
average_ratings.rename(columns={'rating': 'avg_rating'}, inplace=True)

# print(average_ratings)
average_ratings.head()

,movieId,avg_rating
0,1,3.9
1,2,3.4
2,3,3.3
3,4,2.4
4,5,3.1


In [284]:
# Merge on 'movieId' to add the 'avg_rating' column
# 'how="left"' ensures you keep all rows from your original movies dataframe
movies = pd.merge(movies, average_ratings[['movieId', 'avg_rating']], on='movieId', how='left')

movies.head()

,movieId,title,genres,avg_rating
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,3.9
1,2,Jumanji (1995),Adventure Children Fantasy,3.4
2,3,Grumpier Old Men (1995),Comedy Romance,3.3
3,4,Waiting to Exhale (1995),Comedy Drama Romance,2.4
4,5,Father of the Bride Part II (1995),Comedy,3.1


In [285]:
# Extract movie name and year into separate columns
movies[['title', 'year']] = movies['title'].str.extract(r'^(.*?)\s*\((\d{4})\)$')

# Clean up whitespace if needed
movies['title'] = movies['title'].str.strip()

movies.head()

,movieId,title,genres,avg_rating,year
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,3.9,1995
1,2,Jumanji,Adventure Children Fantasy,3.4,1995
2,3,Grumpier Old Men,Comedy Romance,3.3,1995
3,4,Waiting to Exhale,Comedy Drama Romance,2.4,1995
4,5,Father of the Bride Part II,Comedy,3.1,1995


In [286]:
# Merge the imdbId column from links into movies based on 'movieId'
movies = pd.merge(movies, links[['movieId', 'imdbId']], on='movieId', how='left')

In [287]:
movies.head()

,movieId,title,genres,avg_rating,year,imdbId
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,3.9,1995,114709
1,2,Jumanji,Adventure Children Fantasy,3.4,1995,113497
2,3,Grumpier Old Men,Comedy Romance,3.3,1995,113228
3,4,Waiting to Exhale,Comedy Drama Romance,2.4,1995,114885
4,5,Father of the Bride Part II,Comedy,3.1,1995,113041


**Compute Similarity Scores**

In [288]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer


In [289]:
# Convert text to vectors
vectorizer = CountVectorizer()
vector_matrix = vectorizer.fit_transform(movies['genres'])
# print(vector_matrix)

similarity_matrix = cosine_similarity(vector_matrix)

# Format the output into a readable DataFrame (Optional)
df_similarity = pd.DataFrame(
    similarity_matrix,
    columns=[f"Doc {i+1}" for i in range(len(movies['genres']))],
    index=[f"Doc {i+1}" for i in range(len(movies['genres']))]
)

# print("--- Cosine Similarity Matrix ---")
# print(df_similarity)

In [290]:
# Check the type
if isinstance(similarity_matrix , pd.DataFrame):
    print("The variable is a DataFrame!")
else:
    print("The variable is NOT a DataFrame.")

The variable is NOT a DataFrame.


In [291]:
movies.head(17)

,movieId,title,genres,avg_rating,year,imdbId
0,1,Toy Story,Adventure Animation Children Comedy Fantasy,3.9,1995,114709
1,2,Jumanji,Adventure Children Fantasy,3.4,1995,113497
2,3,Grumpier Old Men,Comedy Romance,3.3,1995,113228
3,4,Waiting to Exhale,Comedy Drama Romance,2.4,1995,114885
4,5,Father of the Bride Part II,Comedy,3.1,1995,113041
5,6,Heat,Action Crime Thriller,3.9,1995,113277
6,7,Sabrina,Comedy Romance,3.2,1995,114319
7,8,Tom and Huck,Adventure Children,2.9,1995,112302
8,9,Sudden Death,Action,3.1,1995,114576
9,10,GoldenEye,Action Adventure Thriller,3.5,1995,113189


In [292]:
def get_movie_recommendations(idx, cosine_sim_matrix, movies, top_n):
    """
    Recommends top_n movies similar to the given movie index based on a cosine similarity matrix.

    Parameters:
    idx (int): The index of the movie you want recommendations for.
    cosine_sim_matrix (np.ndarray or pd.DataFrame): The precomputed cosine similarity matrix.
    df (pd.DataFrame): DataFrame containing the 'title' column.
    top_n (int): Number of recommendations to return.

    Returns:
    list: Titles of the recommended movies.
    """

    # Get the pairwise similarity scores for all movies with this movie
    # Note: If cosine_sim_matrix is a DataFrame, use .iloc[idx]
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Sort the movies based on the similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # sim_scores: A sorted list of tuples (index, score)

    # Get the scores of the top_n most similar movies
    # (Skip the first one since it's the movie itself, which has a similarity of 1.0)
    top_indices = []
    i=0
    while len(top_indices)!=top_n:
        idx1 = sim_scores[i][0]
        if  movies['title'].iloc[idx] != movies['title'].iloc[idx1]:
            top_indices.append(idx1)
        i+=1

    # top_indices = [i[0] for i in sim_scores[0:top_n + 1]]
    # print(top_indices)
    return  top_indices


# **Predictions**

In [293]:
movie_name = "Shanghai Triad (Yao a yao yao dao waipo qiao)"
# 1. Check if the movie exists in the dataframe
matching_movies = movies[movies['title'].str.lower() == movie_name.lower()]
if matching_movies.empty:
    print(f"Error: '{movie_name}' not found in the dataset.")
else:
    # Get the index of the movie
    idx = matching_movies.index[0]
    print(idx)
    movie_genre = movies['genres'].iloc[idx]
    movie_year = movies['year'].iloc[idx]
    movie_avg_rating = movies['avg_rating'].iloc[idx]

    print(f"Movie: {movie_name}")
    print(f"Genre: {movie_genre}")
    # print(f"Year: {movie_year}")
    # print(f"Average Rating: {movie_avg_rating}")

    indices = get_movie_recommendations(idx, similarity_matrix, movies, 4)
    recommendations_titles = movies['title'].iloc[indices].tolist()
    recommendations_genres = movies['genres'].iloc[indices].tolist()
    recommendations_avg_ratings = movies['avg_rating'].iloc[indices].tolist()
    recommendations_release_date = movies['year'].iloc[indices].tolist()
    # print("Recommended movies:")
    print(recommendations_titles)
    print(recommendations_genres)
    # print(recommendations_avg_ratings)
    # print(recommendations_avg_ratings)


29
Movie: Shanghai Triad (Yao a yao yao dao waipo qiao)
Genre: Crime Drama
['Casino', 'Dead Man Walking', 'Hate (Haine, La)', "Young Poisoner's Handbook, The"]
['Crime Drama', 'Crime Drama', 'Crime Drama', 'Crime Drama']
